# Preliminary Med Gemma

**Purpose.** Exploratory MedGemma experiment (not used in the thesis results).

**Thesis reference.** provenance

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [14]:
import os
import random
import mlflow
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
import xgboost as xgb
from xgboost.callback import EarlyStopping as XGBoostEarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import json
import itertools
import torch

from huggingface_hub import login
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

In [8]:
diab_wo_ckd = ["P0067", "P0187", "P0231", "P0486", "P0488", "P0508", "P0510", "P0511", "P0556", "P0603", "P0609",
               "P0744", "P0767", "P0778", "P0784", "P0800", "P0832", "P0834", "P0858", "P0863", "P0869", "P0876",
               "P0895", "P0934", "P0938", "P0950", "P0960", "P0961", "P0963", "P0964", "P0967", "P0969", "P0976",
               "P0981", "P1000", "P1004", "P1009", "P1010", "P1011", "P1015", "P1024", "P1030", "P1032", "P1033"]

diab_ckd1 = ["P0045", "P0098", "P0110", "P0210", "P0279", "P0309", "P0316", "P0332", "P0375", "P0470", "P0598", "P0605",
             "P0690", "P0753", "P0783", "P0819", "P0909", "P0922", "P0925", "P0948", "P0965", "P0966", "P0968", "P0974",
             "P1007", "P1008", "P1012", "P1018", "P1021", "P1025", "P1028", "P1029"]

diab_ckd2 = ["P0004", "P0010", "P0013", "P0069", "P0070", "P0079", "P0080", "P0083", "P0103", "P0143", "P0160", "P0161",
             "P0175", "P0181", "P0199", "P0203", "P0205", "P0213", "P0214", "P0223", "P0226", "P0238", "P0240", "P0246",
             "P0260", "P0261", "P0285", "P0318", "P0327", "P0330", "P0341", "P0358", "P0362", "P0364", "P0369", "P0383",
             "P0391", "P0399", "P0411", "P0419", "P0447", "P0459", "P0463", "P0466", "P0480", "P0483", "P0490", "P0495",
             "P0497", "P0499", "P0506", "P0518", "P0558", "P0600", "P0611", "P0614", "P0618", "P0620", "P0624", "P0628",
             "P0631", "P0654", "P0662", "P0672", "P0679", "P0689", "P0699", "P0708", "P0711", "P0715", "P0729", "P0759",
             "P0775", "P0787", "P0795", "P0804", "P0822", "P0826", "P0830", "P0848", "P0859", "P0873", "P0882", "P0883",
             "P0884", "P0903", "P0911", "P0919", "P0932", "P0946", "P0970", "P0972", "P0975", "P0977", "P0994", "P0998",
             "P1006", "P1013", "P1016", "P1017", "P1022", "P1026"]

diab_ckd3 = ["P0009", "P0012", "P0101", "P0115", "P0121", "P0137", "P0138", "P0172", "P0179", "P0182", "P0184", "P0194",
             "P0202", "P0221", "P0242", "P0265", "P0266", "P0269", "P0288", "P0310", "P0317", "P0326", "P0328", "P0334",
             "P0337", "P0342", "P0360", "P0376", "P0378", "P0385", "P0402", "P0404", "P0413", "P0414", "P0416", "P0425",
             "P0441", "P0452", "P0465", "P0476", "P0477", "P0485", "P0515", "P0523", "P0536", "P0545", "P0547", "P0550",
             "P0551", "P0568", "P0571", "P0572", "P0577", "P0602", "P0612", "P0621", "P0622", "P0634", "P0636", "P0645",
             "P0647", "P0660", "P0661", "P0663", "P0665", "P0667", "P0670", "P0680", "P0691", "P0692", "P0701", "P0702",
             "P0709", "P0748", "P0752", "P0821", "P0825", "P0843", "P0868", "P0880", "P0929", "P0973", "P1014", "P1027"]

diab_ckd4 = ["P0118", "P0222", "P0361", "P0445", "P0451", "P0656", "P0657", "P0760"]

diabetes_patients_non_ckd = ['P0067', 'P0187', 'P0231', 'P0486', 'P0488', 'P0508', 'P0510', 'P0511', 'P0556', 'P0603', 'P0609',
                     'P0744', 'P0767', 'P0778', 'P0784', 'P0800', 'P0832', 'P0834', 'P0858', 'P0863', 'P0869', 'P0876',
                     'P0895', 'P0934', 'P0938', 'P0950', 'P0960', 'P0961', 'P0963', 'P0964', 'P0967', 'P0969', 'P0976',
                     'P0981', 'P1000', 'P1004', 'P1009', 'P1010', 'P1011', 'P1015', 'P1024', 'P1030', 'P1032', 'P1033']

diabetes_patients_ckd = ['P0009', 'P0012', 'P0101', 'P0115', 'P0121', 'P0137', 'P0138', 'P0172', 'P0179', 'P0182',
                         'P0184', 'P0194', 'P0202', 'P0221', 'P0242', 'P0265', 'P0266', 'P0269', 'P0288', 'P0310',
                         'P0317', 'P0326', 'P0328', 'P0334', 'P0337', 'P0342', 'P0360', 'P0376', 'P0378', 'P0385',
                         'P0402', 'P0404', 'P0413', 'P0414', 'P0416', 'P0425', 'P0441', 'P0452', 'P0465', 'P0476',
                         'P0477', 'P0485', 'P0515', 'P0523', 'P0536', 'P0545', 'P0547', 'P0550', 'P0551', 'P0568',
                         'P0571', 'P0572', 'P0577', 'P0602', 'P0612', 'P0621', 'P0622', 'P0634', 'P0636', 'P0645',
                         'P0647', 'P0660', 'P0661', 'P0663', 'P0665', 'P0667', 'P0670', 'P0680', 'P0691', 'P0692',
                         'P0701', 'P0702', 'P0709', 'P0748', 'P0752', 'P0821', 'P0825', 'P0843', 'P0868', 'P0880',
                         'P0929', 'P0973', 'P1014', 'P1027', 'P0118', 'P0222', 'P0361', 'P0445', 'P0451', 'P0656',
                         'P0657', 'P0760']

non_diabetes_patients_non_ckd = ["P0038", "P0047", "P0048", "P0062", "P0126", "P0140", "P0157", "P0159", "P0166", "P0248",
                         "P0252", "P0254", "P0267", "P0290", "P0304", "P0340", "P0392", "P0396", "P0421", "P0450",
                         "P0479", "P0484", "P0509", "P0512", "P0513", "P0526", "P0546", "P0626", "P0648", "P0735",
                         "P0745", "P0758", "P0769", "P0771", "P0782", "P0786", "P0794", "P0806", "P0809", "P0813",
                         "P0823", "P0831", "P0835", "P0853", "P0861", "P0864", "P0872", "P0877", "P0886", "P0887",
                         "P0890", "P0896", "P0901", "P0904", "P0905", "P0907", "P0924", "P0926", "P0940", "P0945",
                         "P0947", "P1001"]

non_diabetes_patients_ckd = ["P0005", "P0009", "P0012", "P0018", "P0022", "P0052", "P0055", "P0068", "P0085", "P0092",
                             "P0099", "P0101", "P0115", "P0120", "P0121", "P0124", "P0125", "P0133", "P0135", "P0137",
                             "P0138", "P0153", "P0155", "P0167", "P0168", "P0172", "P0179", "P0180", "P0182", "P0184",
                             "P0185", "P0194", "P0196", "P0201", "P0202", "P0211", "P0221", "P0225", "P0236", "P0237",
                             "P0239", "P0242", "P0247", "P0265", "P0266", "P0269", "P0276", "P0288", "P0300", "P0310",
                             "P0312", "P0317", "P0319", "P0325", "P0326", "P0328", "P0334", "P0337", "P0342", "P0345",
                             "P0354", "P0357", "P0360", "P0371", "P0376", "P0377", "P0378", "P0385", "P0397", "P0398",
                             "P0402", "P0404", "P0407", "P0412", "P0413", "P0414", "P0415", "P0416", "P0418", "P0425",
                             "P0429", "P0437", "P0440", "P0441", "P0446", "P0452", "P0465", "P0475", "P0476", "P0477",
                             "P0478", "P0485", "P0487", "P0491", "P0496", "P0514", "P0515", "P0523", "P0536", "P0537",
                             "P0539", "P0540", "P0541", "P0542", "P0545", "P0547", "P0550", "P0551", "P0553", "P0561",
                             "P0568", "P0571", "P0572", "P0574", "P0577", "P0586", "P0589", "P0595", "P0596", "P0602",
                             "P0612", "P0616", "P0621", "P0622", "P0634", "P0635", "P0636", "P0642", "P0645", "P0647",
                             "P0651", "P0653", "P0659", "P0660", "P0661", "P0663", "P0665", "P0667", "P0668", "P0670",
                             "P0671", "P0675", "P0680", "P0691", "P0692", "P0693", "P0698", "P0701", "P0702", "P0709",
                             "P0731", "P0732", "P0747", "P0748", "P0752", "P0774", "P0776", "P0779", "P0781", "P0789",
                             "P0790", "P0807", "P0821", "P0825", "P0833", "P0836", "P0843", "P0849", "P0868", "P0880",
                             "P0898", "P0906", "P0910", "P0929", "P0973", "P1014", "P1023", "P1027", "P0118", "P0222",
                             "P0333", "P0361", "P0381", "P0445", "P0451", "P0503", "P0516", "P0527", "P0656", "P0657",
                             "P0681", "P0751", "P0760", "P0597"]

In [10]:
import os
import shutil

source_folder = os.path.join("images", "recurrence_plot") 
destination_folder =  os.path.join("images", "recurrence_plot_filtered")   
ids =  set(non_diabetes_patients_non_ckd + diabetes_patients_non_ckd 
           + non_diabetes_patients_ckd + diabetes_patients_ckd)

for img_id in ids:
    filename = f"{img_id}.png"
    src_path = os.path.join(source_folder, filename)
    dest_path = os.path.join(destination_folder, filename)
    print(src_path)
    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)
        print(f"Copied: {filename}")
    else:
        print(f"Not found: {filename}")


images/recurrence_plot/P0784.png
Copied: P0784.png
images/recurrence_plot/P0863.png
Copied: P0863.png
images/recurrence_plot/P0225.png
Not found: P0225.png
images/recurrence_plot/P0404.png
Not found: P0404.png
images/recurrence_plot/P0510.png
Copied: P0510.png
images/recurrence_plot/P0398.png
Not found: P0398.png
images/recurrence_plot/P0101.png
Copied: P0101.png
images/recurrence_plot/P0231.png
Not found: P0231.png
images/recurrence_plot/P0834.png
Copied: P0834.png
images/recurrence_plot/P0416.png
Not found: P0416.png
images/recurrence_plot/P0545.png
Copied: P0545.png
images/recurrence_plot/P0153.png
Copied: P0153.png
images/recurrence_plot/P0537.png
Copied: P0537.png
images/recurrence_plot/P0660.png
Copied: P0660.png
images/recurrence_plot/P0550.png
Copied: P0550.png
images/recurrence_plot/P0597.png
Copied: P0597.png
images/recurrence_plot/P0869.png
Copied: P0869.png
images/recurrence_plot/P1014.png
Copied: P1014.png
images/recurrence_plot/P0376.png
Not found: P0376.png
images/recurr

In [7]:
label_dict = {}

for pid in non_diabetes_patients_non_ckd + diabetes_patients_non_ckd:
    label_dict[pid] = 0
for pid in non_diabetes_patients_ckd + diabetes_patients_ckd:
    label_dict[pid] = 1
    
num_ckd_positive = sum(1 for label in label_dict.values() if label == 1)
num_ckd_negative = sum(1 for label in label_dict.values() if label == 0)
print(f"  CKD Positive (Label 1): {num_ckd_positive} patients")
print(f"  CKD Negative (Label 0): {num_ckd_negative} patients")

  CKD Positive (Label 1): 194 patients
  CKD Negative (Label 0): 106 patients


In [8]:
DATA_ROOT = "../ahen_data/filtered_tsv_2"
IMAGE_ROOT = "images"

IMAGE_TYPES = {
    "grayscale_heatmap": os.path.join(IMAGE_ROOT, "grayscale_heatmap"),
    "pseudocolor_heatmap": os.path.join(IMAGE_ROOT, "pseudocolor_heatmap"),
    "rgb_mz_split": os.path.join(IMAGE_ROOT, "rgb_mz_split"),
    "lineplot_intensity": os.path.join(IMAGE_ROOT, "lineplot_intensity"),
    "scatterplot": os.path.join(IMAGE_ROOT, "scatterplot"),
    "tic_plot": os.path.join(IMAGE_ROOT, "tic_plot"),
    "recurrence_plot": os.path.join(IMAGE_ROOT, "recurrence_plot"),
}

In [9]:
NGROK_FORWARDING_URL = " https://b42b-46-217-30-1.ngrok-free.app"  
MLFLOW_TRACKING_URI = NGROK_FORWARDING_URL

print(f"NGROK_FORWARDING_URL: {NGROK_FORWARDING_URL}")
print(f"MLFLOW_TRACKING_URI set to: {MLFLOW_TRACKING_URI}")


try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"Successfully set tracking URI. mlflow.get_tracking_uri() reports: {mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"Error setting tracking URI: {e}")
    raise

NGROK_FORWARDING_URL:  https://b42b-46-217-30-1.ngrok-free.app
MLFLOW_TRACKING_URI set to:  https://b42b-46-217-30-1.ngrok-free.app
Successfully set tracking URI. mlflow.get_tracking_uri() reports:  https://b42b-46-217-30-1.ngrok-free.app


In [11]:
mlflow.set_experiment("CKD_med_gemma")

2025/06/25 11:32:45 INFO mlflow.tracking.fluent: Experiment with name 'CKD_med_gemma' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlflow/artifact_store/264096057374306859', creation_time=1750851168585, experiment_id='264096057374306859', last_update_time=1750851168585, lifecycle_stage='active', name='CKD_med_gemma', tags={}>

In [18]:
# pip install -q -U transformers accelerate bitsandbytes salesforce-lavis
pip install -q -U transformers accelerate bitsandbytes sentencepiece


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
salesforce-lavis 1.0.2 requires transformers<4.27,>=4.25.0, but you have transformers 4.52.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [19]:
import json
import torch
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

all_patient_ids = list(label_dict.keys())
print(f"Found {len(all_patient_ids)} total patients in label_dict.")

print("\nSetting up the Vision-Language Model (BLIP)")
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
caption_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)
print(f"BLIP model loaded on {device}.")

patient_reports = {}
print(f"\nGenerating reports for {len(all_patient_ids)} patients...")

for pid in tqdm(all_patient_ids, desc="Processing Patients"):
    full_report = []
    # Generate a caption for each image type
    for img_type, img_dir in IMAGE_TYPES.items():
        img_path = os.path.join(img_dir, f"{pid}.png")
        if not os.path.exists(img_path):
            continue

        try:
            raw_image = Image.open(img_path).convert("RGB")
            inputs = processor(images=raw_image, return_tensors="pt").to(device)
            output_ids = caption_model.generate(**inputs, max_length=75, num_beams=5, early_stopping=True)
            caption = processor.decode(output_ids[0], skip_special_tokens=True).strip()
            report_line = f"Analysis of {img_type.replace('_', ' ')}: {caption}."
            full_report.append(report_line)

        except Exception as e:
            print(f"Warning: Could not process image {img_path}. Error: {e}")

    if full_report:
        patient_reports[pid] = " ".join(full_report)
    else:
        print(f"Warning: No images found for patient {pid}. This patient will be skipped in training.")

output_data = {
    "labels": label_dict,
    "reports": patient_reports
}
output_file = "training_data.json"
with open(output_file, 'w') as f:
    json.dump(output_data, f, indent=4)

print(f"\nSuccessfully generated and saved patient reports and labels to '{output_file}'")
print("\nExample report for one patient:")
example_pid = list(patient_reports.keys())[0] if patient_reports else "None"
if example_pid != "None":
    print(f"Patient ID: {example_pid}")
    print(f"Label: {label_dict[example_pid]}")
    print(f"Report: {patient_reports[example_pid]}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Found 300 total patients in label_dict.

Setting up the Vision-Language Model (BLIP)


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

... (16 outputs left out)

Processing Patients:  62%|██████▏   | 186/300 [17:57<01:50,  1.03it/s]

Processing Patients:  95%|█████████▌| 285/300 [31:06<02:21,  9.41s/it]

Processing Patients: 100%|██████████| 300/300 [32:22<00:00,  6.47s/it]


Successfully generated and saved patient reports and labels to 'training_data.json'

Example report for one patient:
Patient ID: P0038
Label: 0
Report: Analysis of grayscale heatmap: this is a black and white photo of a man on a skateboard. Analysis of pseudocolor heatmap: this is an image of a close up of a blue background. Analysis of rgb mz split: this is a picture of a blurry image of a street light. Analysis of lineplot intensity: a black and white photo of a number of tall poles. Analysis of scatterplot: there is a black and white photo of a person in the dark. Analysis of tic plot: an image of a graph of a line with a clock on top of it. Analysis of recurrence plot: this is an image of a grid of squares in black and white.


In [ ]:
import json
import itertools
import torch

from huggingface_hub import login
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

pipe = pipeline(
    "image-text-to-text",
    model="google/medgemma-4b-it",
    torch_dtype=torch.bfloat16,
    device="cpu",
    trust_remote_code=True,
    use_auth_token=True
)

def run_experiments():
    login(token="")
    with open("training_data.json", 'r') as f:
        data = json.load(f)

    label_dict = data["labels"]
    patient_reports = data["reports"]

    valid_patient_ids = list(patient_reports.keys())
    texts = [patient_reports[pid] for pid in valid_patient_ids]
    labels = [label_dict[pid] for pid in valid_patient_ids]

    print(f"Loaded {len(texts)} patients with valid reports for training.")

    X_train, X_val, y_train, y_val = train_test_split(
        texts,
        labels,
        test_size=0.30,
        random_state=42,
        stratify=labels
    )
    print(f"Training set size: {len(X_train)}, Validation set size: {len(X_val)}")

    param_grid = {
        "model_name": ["google/medgemma-4b-it"],  # Gemma3-based 
        "learning_rate": [5e-5, 3e-5, 1e-5],
        "batch_size": [4, 8, 16],
        "epochs": [3, 5, 7],
    }
    keys, values = zip(*param_grid.items())
    run_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    print(f"\nStarting experiments for {len(run_combinations)} parameter combinations.")

    for params in run_combinations:
        train_and_log_experiment(params, X_train, y_train, X_val, y_val)

    print("\nAll experiments completed.")


def train_and_log_experiment(params, X_train, y_train, X_val, y_val):
    model_name = params["model_name"]
    num_labels = len(set(y_train + y_val))

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        trust_remote_code=True,
        use_auth_token=True
    )
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
        trust_remote_code=True,
        use_auth_token=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        trust_remote_code=True,
        use_auth_token=True
    )

    def tokenize_fn(texts, labels=None):
        encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=512
        )
        if labels is not None:
            encodings["labels"] = labels
        return encodings

    train_enc = tokenize_fn(X_train, y_train)
    val_enc = tokenize_fn(X_val, y_val)

    from datasets import Dataset
    train_dataset = Dataset.from_dict(train_enc)
    val_dataset = Dataset.from_dict(val_enc)

    run_name = (
        f"{model_name.split('/')[-1]}"
        f"_lr{params['learning_rate']}"
        f"_bs{params['batch_size']}"
        f"_ep{params['epochs']}"
    )

    training_args = TrainingArguments(
        output_dir=f"./results/{run_name}",
        run_name=run_name,
        per_device_train_batch_size=params['batch_size'],
        per_device_eval_batch_size=params['batch_size'],
        learning_rate=params['learning_rate'],
        num_train_epochs=params['epochs'],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_dir='./logs',
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
    )

    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='weighted')
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    metrics = trainer.evaluate()
    print(f"Run: {run_name} -> Val Accuracy: {metrics['eval_accuracy']:.4f}, F1: {metrics['eval_f1']:.4f}")


if __name__ == "__main__":
    run_experiments()


In [ ]:


# Gather examples (one per patient, you can extend to multiple images if desired)
training_examples = []
for pid, lbl in label_dict.items():
    # pick one representative image folder (e.g. grayscale_heatmap)
    img_path = os.path.join(IMAGE_TYPES["grayscale_heatmap"], f"{pid}.png")
    if not os.path.exists(img_path):
        # you could loop IMAGE_TYPES here to include a fallback
        continue
    training_examples.append({
        "image": img_path,
        "label": lbl
    })

# Write out JSON
out_file = "training_data.json"
with open(out_file, "w") as f:
    json.dump(training_examples, f, indent=2)

print(f"✅ Saved {len(training_examples)} examples → {out_file}")
